# EDA: Customer Churn Dataset
## Análisis Exhaustivo para Modelamiento Predictivo

In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

with open('customer_churn_data.json') as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f'Shape: {df.shape}')
print(df.head())

Shape: (3000, 200)
   customer_id  signal_0  signal_1  signal_2   noise_0   noise_1   noise_2  \
0            1  1.227542 -2.682519 -2.719936 -0.765884  1.772437  0.729590   
1            2  1.734012 -0.613436  2.263493  0.455903 -1.376789  0.093387   
2            3  0.335140  3.108565  0.844029  0.303693  1.295508  0.201326   
3            4  2.324519  2.739562 -0.433430  1.586197  0.737787 -0.382652   
4            5 -1.079182  1.468852  3.734463 -0.041680  1.566979 -1.052442   

    noise_3   noise_4   noise_5  ...  extreme_1  extreme_2  extreme_3  \
0 -0.776736  2.897539 -1.623317  ...   0.554114  -0.027111   0.399628   
1 -0.303266  0.441905  0.119372  ...  -0.913437   0.292195   0.502724   
2  1.172864 -1.862141 -0.495880  ...  -0.019887  -2.680827   1.993237   
3 -0.547717  1.175436  0.359624  ...   0.849454  -0.560492   0.867877   
4  1.680500 -0.387151  0.716577  ...  -1.214891  -2.575036  -0.602402   

   extreme_4  extreme_5  extreme_6  extreme_7  extreme_8  extreme_9  chur

## ANÁLISIS DE NULOS

In [2]:
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Features con >50% nulos:')
print(missing_pct[missing_pct > 50])
print(f'Total con >75% nulos: {(missing_pct > 75).sum()}')

Features con >50% nulos:
sparse_12    85.966667
sparse_19    85.433333
sparse_16    85.366667
sparse_2     85.333333
sparse_6     85.233333
sparse_17    83.900000
sparse_4     83.333333
sparse_14    83.133333
sparse_7     82.600000
sparse_11    81.900000
sparse_10    81.833333
sparse_13    80.533333
sparse_15    80.133333
sparse_5     80.033333
sparse_0     79.900000
sparse_18    79.800000
sparse_9     79.066667
sparse_1     78.533333
sparse_8     78.466667
sparse_3     76.633333
dtype: float64
Total con >75% nulos: 20


## DISTRIBUCIÓN TARGET

In [3]:
print(df['churn_90d'].value_counts())
print('\nProporción:')
print(df['churn_90d'].value_counts(normalize=True))

churn_90d
0    2647
1     353
Name: count, dtype: int64

Proporción:
churn_90d
0    0.882333
1    0.117667
Name: proportion, dtype: float64


## VARIANZA DE FEATURES

In [4]:
numeric = df.select_dtypes(include=[np.number]).drop('churn_90d', axis=1).columns.tolist()
variance = df[numeric].var().sort_values()
print(f'Total features: {len(numeric)}')
print(f'Features con var < 1: {(variance < 1).sum()}')
print('\nTop 15 menor varianza:')
print(variance.head(15))

Total features: 199
Features con var < 1: 96

Top 15 menor varianza:
constant_19    0.087751
constant_9     0.087852
constant_14    0.088426
constant_22    0.088959
constant_0     0.089244
constant_2     0.089245
constant_13    0.089301
constant_1     0.089330
constant_16    0.089454
constant_20    0.089618
constant_5     0.089874
constant_4     0.090050
constant_10    0.090285
constant_7     0.090808
constant_21    0.090809
dtype: float64


## CORRELACIONES CON CHURN

In [5]:
corr = df[numeric + ['churn_90d']].corr()['churn_90d'].drop('churn_90d').abs().sort_values(ascending=False)
print('Top 20 correlacionadas:')
print(corr.head(20))
print(f'\nRuido puro (|corr| < 0.01): {(corr < 0.01).sum()}')
print(f'Poco poder (|corr| < 0.05): {(corr < 0.05).sum()}')

Top 20 correlacionadas:
signal_2          0.122907
sparse_17         0.098505
sparse_16         0.086482
signal_1          0.067020
sparse_5          0.065577
signal_0          0.065210
sparse_19         0.058655
constant_0        0.047013
sparse_3          0.042009
noise_1           0.041369
collinear_15_b    0.039953
collinear_15_a    0.039931
sparse_2          0.039638
sparse_10         0.038732
collinear_26_b    0.038105
collinear_26_a    0.038085
sparse_8          0.037578
noise_37          0.037511
noise_77          0.036681
constant_23       0.036305
Name: churn_90d, dtype: float64

Ruido puro (|corr| < 0.01): 72
Poco poder (|corr| < 0.05): 190


## COLINEALIDAD

In [6]:
corr_matrix = df[numeric].corr()
high_corr = 0
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.95:
            high_corr += 1

print(f'Pares con |corr| > 0.95: {high_corr}')

Pares con |corr| > 0.95: 30


## OUTLIERS EXTREMOS

In [7]:
extreme_count = 0
for f in numeric:
    if df[f].max() > 1e6:
        print(f'{f}: max={df[f].max():.2e}')
        extreme_count += 1

print(f'\nTotal features con max > 1e6: {extreme_count}')

extreme_0: max=9.97e+08
extreme_1: max=7.77e+08
extreme_2: max=9.05e+08
extreme_3: max=9.77e+08
extreme_4: max=9.94e+08
extreme_5: max=8.51e+08
extreme_6: max=8.88e+08
extreme_7: max=9.27e+08
extreme_8: max=9.46e+08
extreme_9: max=9.91e+08

Total features con max > 1e6: 10


## RESUMEN Y RECOMENDACIONES

In [8]:
print('PROBLEMAS DETECTADOS:')
print(f'- Ruido puro: {(corr < 0.01).sum()} features sin correlacion')
print(f'- Nulos masivos: {(missing_pct > 75).sum()} features con >75% nulos')
print(f'- Baja varianza: {(variance < 1).sum()} features con var < 1')
print(f'- Colinealidad: {high_corr} pares altamente correlacionados')
print(f'- Outliers: {extreme_count} features con max > 1e6')
print('\nRECOMENDACIONES:')
print('1. Eliminar features con >80% nulos')
print('2. Eliminar features con varianza < 1')
print('3. Eliminar features con |corr| < 0.01')
print('4. Eliminar features colineales')
print('5. Usar RobustScaler para outliers')
print('6. Usar GradientBoosting + regularizacion fuerte')

PROBLEMAS DETECTADOS:
- Ruido puro: 72 features sin correlacion
- Nulos masivos: 20 features con >75% nulos
- Baja varianza: 96 features con var < 1
- Colinealidad: 30 pares altamente correlacionados
- Outliers: 10 features con max > 1e6

RECOMENDACIONES:
1. Eliminar features con >80% nulos
2. Eliminar features con varianza < 1
3. Eliminar features con |corr| < 0.01
4. Eliminar features colineales
5. Usar RobustScaler para outliers
6. Usar GradientBoosting + regularizacion fuerte
